In [1]:
# Feature Engineering (Python)

# Tiers
"""
1. Geospatial
2. Rolling behavioral
3. Per-customer distributions
4. Derived ratios / risk scores
5. Graph features
6. Model-prep (log transforms + interactions)
"""

'\n1. Geospatial\n2. Rolling behavioral\n3. Per-customer distributions\n4. Derived ratios / risk scores\n5. Graph features\n6. Model-prep (log transforms + interactions)\n'

In [ ]:
import numpy as np
import pandas as pd

In [3]:
# Sample dataset - small, synthetic to run without full pipeline

In [4]:
df = pd.DataFrame({
    'trans_num': list(range(12)),
    'cc_num': [1,1,1,1, 2,2,2,2, 3,3,3,3],
    'unix_time': [
        # customer 1 - spread over ~2 months
        1569000000, 1569003600, 1569090000, 1574000000,
        # customer 2 - rapid-fire transactions (velocity burst)
        1569000000, 1569000300, 1569000600, 1571000000,
        # customer 3 - sparse
        1569000000, 1570000000, 1575000000, 1577000000,
    ],
    'amt': [50, 200, 45, 600,   30, 25, 28, 400,   75, 80, 70, 1200],
    # home coordinates (customer-level, same per card)
    'cust_lat': [40.71]*4 + [34.05]*4 + [51.50]*4,
    'cust_long': [-74.00]*4 + [-118.24]*4 + [-0.12]*4,
    # merchant coordinates
    'merch_lat': [40.73, 40.80, 40.65, 42.36,   34.10, 34.12, 34.09, 36.17,   51.52, 51.55, 51.48, 48.85],
    'merch_long': [-73.99,-74.01,-74.10,-71.06,  -118.20,-118.22,-118.25,-115.14,  -0.10, -0.09,-0.14, 2.35],
    'merchant': ['m_a','m_b','m_a','m_c', 'm_d','m_d','m_e','m_f', 'm_g','m_g','m_h','m_i'],
    'MerchantCategory': ['grocery','grocery','gas','travel', 'grocery','grocery','gas','travel', 'grocery','grocery','gas','travel'],
    'TransactionHour': [10,11,9,22,  8,8,8,14,  12,13,10,3],
    'TimeSinceLastTxnSec': [0,3600,86400,4000000, 0,300,300,2000000, 0,1000000,5000000,2000000],
    'TxnCount_1h': [1,1,1,1, 1,2,3,1, 1,1,1,1],
    'TxnCount_24h': [1,2,3,1, 1,3,4,1, 1,1,1,1],
    'TxnAmount_1h': [50,200,45,600, 30,55,83,400, 75,80,70,1200],
    'TxnAmount_7d': [50,250,295,600, 30,55,83,400, 75,155,225,1350],
    'AmountZScore': [0.1,-0.2,0.3,2.5, -0.1,0.0,-0.1,3.0, 0.2,0.1,0.0,4.0],
    'IPRiskScore': [10,10,10,80, 5,5,5,90, 20,20,20,70],
    'EmailRiskScore': [5,5,5,40,   5,5,5,60,  10,10,10,50],
    'PhoneRiskScore': [2,2,2,20,   2,2,2,30,   5,5,5,25],
    'AddressRiskScore': [3,3,3,15,   3,3,3,25,   8,8,8,20],
    'BotLikelihoodScore': [0,0,0,50,   0,0,0,70,   5,5,5,60],
    'MerchantRiskScore': [5,5,5,30,   5,5,5,40,  10,10,10,35],
    'DeviceTrustScore': [90,90,90,20, 95,95,95,10, 85,85,85,15],
    'HistoricalVelocityScore': [2,2,2,30, 2,2,2,40, 5,5,5,35],
    'IsNewDevice': [0,0,0,1, 0,0,0,1, 0,0,0,1],
    'IsVPN': [0,0,0,1, 0,0,0,1, 0,0,0,1],
    'NumAccountsOnDevice': [1,1,1,3, 1,1,1,4, 1,1,1,2],
    'PreviousFraudFlag': [0,0,0,0, 0,0,0,1, 0,0,0,0],
    'DaysSincePasswordChange': [np.nan,np.nan,np.nan,2, 90,90,90,0.5, 15,15,15,np.nan],
    'is_fraud': [0,0,0,1, 0,0,0,1, 0,0,0,1],
})

df = df.sort_values(['cc_num','unix_time']).reset_index(drop=True)
df[['trans_num','cc_num','unix_time','amt','merchant','is_fraud']].head(8)

,trans_num,cc_num,unix_time,amt,merchant,is_fraud
0,0,1,1569000000,50,m_a,0
1,1,1,1569003600,200,m_b,0
2,2,1,1569090000,45,m_a,0
3,3,1,1574000000,600,m_c,1
4,4,2,1569000000,30,m_d,0
5,5,2,1569000300,25,m_d,0
6,6,2,1569000600,28,m_e,0
7,7,2,1571000000,400,m_f,1


In [5]:
# 1. GEOSPATIAL FEATURES

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371.0 # Earth radius 
    rl = np.radians
    dlat = rl(lat2) - rl(lat1)
    dlon = rl(lon2) - rl(lon1)
    a = np.sin(dlat/2)**2 + np.cos(rl(lat1)) * np.cos(rl(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

In [ ]:
# Distance from customer home to merchant
df['DistanceFromHome'] = haversine_km(
    df['cust_lat'].values, 
    df['cust_long'].values,
    df['merch_lat'].values, 
    df['merch_long'].values,
    ).round(2)

# Distance from previous merchant location (per customer)
prev_mlat = df.groupby('cc_num')['merch_lat'].shift(1)
prev_mlong = df.groupby('cc_num')['merch_long'].shift(1)

df['DistanceFromLastTxn'] = haversine_km(
    prev_mlat.values, 
    prev_mlong.values,
    df['merch_lat'].values, 
    df['merch_long'].values,
    ).round(2)

df['DistanceFromLastTxn'] = df['DistanceFromLastTxn'].fillna(0)

# Anomalous travel flag (>500 km/h)
hrs = df['TimeSinceLastTxnSec'].replace(0, np.nan) / 3600
speed_kmh = df['DistanceFromLastTxn'] / hrs

df['ImpossibleTravelFlag'] = ((speed_kmh > 500) & hrs.notna()).astype(np.int8)

df[['cc_num','amt','DistanceFromHome','DistanceFromLastTxn','ImpossibleTravelFlag']]

,cc_num,amt,DistanceFromHome,DistanceFromLastTxn,ImpossibleTravelFlag
0,1,50,2.38,0.00,0
1,1,200,10.04,7.96,0
2,1,45,10.75,18.32,0
3,1,600,305.82,316.57,0
4,2,30,6.67,0.00,0
5,2,25,8.00,2.89,0
6,2,28,4.54,4.33,0
7,2,400,367.50,365.31,0
8,3,75,2.62,0.00,0
9,3,80,5.93,3.41,0


In [7]:
# 2. BEHAVIORAL FEATURES (ROLLING)

In [8]:
# AmountDeviation_30d: z-score vs 30-day rolling window
WINDOW_30D = 2_592_000  # 30d * 24h * 60m * 60s = 2,592,000 seconds

results = []
for cc, grp in df.groupby('cc_num', sort=False):
    t = grp['unix_time'].values.astype(np.float64)
    amt = grp['amt'].values.astype(np.float64)
    n = len(grp)
    z = np.zeros(n)
    for i in range(n):
        window = amt[(t >= t[i] - WINDOW_30D) & (np.arange(n) < i)]
        if len(window) >= 2:
            mu, sd = window.mean(), window.std()
            z[i] = (amt[i] - mu) / (sd if sd > 0 else 1.0)
    results.append(pd.Series(z, index=grp.index))

df['AmountDeviation_30d'] = pd.concat(results).round(4)

# SpendBurstScore - 1h spend vs expected hourly from 7d average
hourly_baseline = df['TxnAmount_7d'] / (7 * 24)
df['SpendBurstScore'] = (df['TxnAmount_1h'] / hourly_baseline.replace(0, np.nan)).fillna(0).clip(0, 100).round(4)

# VelocityAcceleration - actual 1h count vs expected from 24h rate
expected_1h = df['TxnCount_24h'] / 24.0
df['VelocityAcceleration'] = (df['TxnCount_1h'] / expected_1h.replace(0, np.nan)).fillna(0).clip(0, 100).round(4)

df[['cc_num','amt','AmountDeviation_30d','SpendBurstScore','VelocityAcceleration']]

,cc_num,amt,AmountDeviation_30d,SpendBurstScore,VelocityAcceleration
0,1,50,0.0000,100.0000,24.0
1,1,200,0.0000,100.0000,12.0
2,1,45,-1.0667,25.6271,8.0
3,1,600,0.0000,100.0000,24.0
4,2,30,0.0000,100.0000,24.0
5,2,25,0.0000,100.0000,16.0
6,2,28,0.2000,100.0000,18.0
7,2,400,181.2013,100.0000,24.0
8,3,75,0.0000,100.0000,24.0
9,3,80,0.0000,86.7097,24.0


In [9]:
# 3. PER-CUSTOMER DISTRIBUTION FEATURES

In [10]:
# TimeOfDayDeviation - how far is this hour from the customer's typical hour?
# (trained on a "training" cutoff; here we just use all rows as proxy)
cust_hour = df.groupby('cc_num')['TransactionHour'].agg(h_mean='mean', h_std='std').reset_index()
cust_hour['h_std'] = cust_hour['h_std'].fillna(1).replace(0, 1)

df = df.merge(cust_hour, on='cc_num', how='left')

df['TimeOfDayDeviation'] = ((df['TransactionHour'] - df['h_mean']) / df['h_std']).round(4)
df.drop(columns=['h_mean','h_std'], inplace=True)



# CategoryDeviationScore - surprise = 1 - fraction of this category in history
cat_counts = df.groupby(['cc_num','MerchantCategory']).size().rename('n')
cust_total = df.groupby('cc_num').size().rename('total')
cat_frac = (cat_counts / cust_total).rename('frac').reset_index()

df = df.merge(cat_frac, on=['cc_num','MerchantCategory'], how='left')

df['CategoryDeviationScore'] = (1 - df['frac'].fillna(1)).round(4)
df.drop(columns=['frac'], inplace=True)



# InterArrivalEntropy - Shannon entropy of time gaps per customer
def shannon_entropy(values):
    if len(values) < 2:
        return 0.0
    counts, _ = np.histogram(values, bins=min(20, len(values)))
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

entropy_map = {
    cc: shannon_entropy(np.diff(grp['unix_time'].values))
    for cc, grp in df.groupby('cc_num', sort=False)
}
df['InterArrivalEntropy'] = df['cc_num'].map(entropy_map).round(4)



# MerchantAmountZScore - z-score vs merchant-level distribution
merch_agg = df.groupby('merchant')['amt'].agg(merch_mu='mean', merch_std='std').reset_index()
merch_agg['merch_std'] = merch_agg['merch_std'].fillna(1).replace(0, 1)

df = df.merge(merch_agg, on='merchant', how='left')

df['MerchantAmountZScore'] = ((df['amt'] - df['merch_mu']) / df['merch_std']).round(4)
df.drop(columns=['merch_mu','merch_std'], inplace=True)

df[['cc_num','TransactionHour','MerchantCategory', 'TimeOfDayDeviation','CategoryDeviationScore', 'InterArrivalEntropy','MerchantAmountZScore']]

,cc_num,TransactionHour,MerchantCategory,TimeOfDayDeviation,CategoryDeviationScore,InterArrivalEntropy,MerchantAmountZScore
0,1,10,grocery,-0.4954,0.50,0.9183,0.7071
1,1,11,grocery,-0.3303,0.50,0.9183,0.0000
2,1,9,gas,-0.6606,0.75,0.9183,-0.7071
3,1,22,travel,1.4863,0.75,0.9183,0.0000
4,2,8,grocery,-0.5000,0.50,0.9183,0.7071
5,2,8,grocery,-0.5000,0.50,0.9183,-0.7071
6,2,8,gas,-0.5000,0.75,0.9183,0.0000
7,2,14,travel,1.5000,0.75,0.9183,0.0000
8,3,12,grocery,0.5544,0.50,0.9183,-0.7071
9,3,13,grocery,0.7762,0.50,0.9183,0.7071


In [11]:
# 4. SCORE FEATURES

In [ ]:
RISK_WEIGHTS = {
    'IPRiskScore': 0.20,
    'EmailRiskScore': 0.15,
    'PhoneRiskScore': 0.10,
    'AddressRiskScore': 0.10,
    'BotLikelihoodScore': 0.20,
    'MerchantRiskScore': 0.10,
    'DeviceTrustScore': 0.10,  # inverted
    'HistoricalVelocityScore': 0.05,
}

score = np.zeros(len(df))
for col, w in RISK_WEIGHTS.items():
    vals = df[col].fillna(0).values.astype(float)
    if col == 'DeviceTrustScore':
        vals = 100.0 - vals # invert: high trust -> low risk
    if vals.max() > 1.5:
        vals = vals / 100.0 # normalize 0-100 -> 0-1
    score += w * vals
df['RiskSignalWeightedScore'] = score.round(4)


# Password-change feats
d = df['DaysSincePasswordChange']
df['NullPwdChange'] = d.isna().astype(np.int8)
df['HasPasswordChange'] = (~d.isna()).astype(np.int8)
df['DaysSincePasswordChangeBucket'] = pd.cut(
    d.fillna(9999),
    bins=[-1, 1, 7, 30, 365, 9999],
    labels=[0, 1, 2, 3, 4],
).astype('Int64')

df[['cc_num','is_fraud','RiskSignalWeightedScore','NullPwdChange','HasPasswordChange','DaysSincePasswordChangeBucket']]

,cc_num,is_fraud,RiskSignalWeightedScore,NullPwdChange,HasPasswordChange,DaysSincePasswordChangeBucket
0,1,0,0.0485,1,0,4
1,1,0,0.0485,1,0,4
2,1,0,0.0485,1,0,4
3,1,1,0.4800,0,1,1
4,2,0,0.0335,0,1,3
5,2,0,0.0335,0,1,3
6,2,0,0.0335,0,1,3
7,2,1,0.6150,0,1,0
8,3,0,0.1055,0,1,2
9,3,0,0.1055,0,1,2


In [13]:
# 5. GRAPH FEATURES

In [14]:
# Without device_registry.csv, fall back to the NumAccountsOnDevice proxy

df['DeviceClusterSize'] = df['NumAccountsOnDevice'].fillna(1).astype(int)
df['LinkedFraudAccounts'] = ( (df['NumAccountsOnDevice'] > 3) & (df['PreviousFraudFlag'] == 1) ).astype(int)


# SharedMerchantFraudRate - fraud rate at the same merchant
merch_stats = (df.groupby('merchant').agg(total=('is_fraud','count'), fraud=('is_fraud','sum')).reset_index())
merch_stats['SharedMerchantFraudRate'] = (merch_stats['fraud'] / merch_stats['total'].replace(0, np.nan)).fillna(0)

df = df.merge(merch_stats[['merchant','SharedMerchantFraudRate']], on='merchant', how='left')

df[['cc_num','merchant','is_fraud','DeviceClusterSize','LinkedFraudAccounts','SharedMerchantFraudRate']]

,cc_num,merchant,is_fraud,DeviceClusterSize,LinkedFraudAccounts,SharedMerchantFraudRate
0,1,m_a,0,1,0,0.0
1,1,m_b,0,1,0,0.0
2,1,m_a,0,1,0,0.0
3,1,m_c,1,3,0,1.0
4,2,m_d,0,1,0,0.0
5,2,m_d,0,1,0,0.0
6,2,m_e,0,1,0,0.0
7,2,m_f,1,4,1,1.0
8,3,m_g,0,1,0,0.0
9,3,m_g,0,1,0,0.0


In [15]:
# 6. LOGS, INTERACTION FEATURES

In [16]:
df['log_amt'] = np.log1p(df['amt'])
df['log_DistanceFromHome'] = np.log1p(df['DistanceFromHome'])
df['log_TimeSinceLastTxn'] = np.log1p(df['TimeSinceLastTxnSec'].fillna(0))
df['log_TxnAmount_7d'] = np.log1p(df['TxnAmount_7d'].fillna(0))

df['IsNewDevice_x_IPRisk'] = (df['IsNewDevice'] * df['IPRiskScore']).round(4)
df['IsVPN_x_AmtZScore'] = (df['IsVPN'] * df['AmountZScore'].abs()).round(4)
df['Bot_x_Velocity'] = (df['BotLikelihoodScore'] * df['TxnCount_1h']).round(4)

df[['cc_num','amt','log_amt','DistanceFromHome','log_DistanceFromHome','IsNewDevice_x_IPRisk','IsVPN_x_AmtZScore','Bot_x_Velocity','is_fraud']]

,cc_num,amt,log_amt,DistanceFromHome,log_DistanceFromHome,IsNewDevice_x_IPRisk,IsVPN_x_AmtZScore,Bot_x_Velocity,is_fraud
0,1,50,3.931826,2.38,1.217876,0,0.0,0,0
1,1,200,5.303305,10.04,2.401525,0,0.0,0,0
2,1,45,3.828641,10.75,2.463853,0,0.0,0,0
3,1,600,6.398595,305.82,5.726261,80,2.5,50,1
4,2,30,3.433987,6.67,2.037317,0,0.0,0,0
5,2,25,3.258097,8.00,2.197225,0,0.0,0,0
6,2,28,3.367296,4.54,1.711995,0,0.0,0,0
7,2,400,5.993961,367.50,5.909441,90,3.0,70,1
8,3,75,4.330733,2.62,1.286474,0,0.0,5,0
9,3,80,4.394449,5.93,1.935860,0,0.0,5,0


In [17]:
# OVERVIEW

In [18]:
PYTHON_FEATURE_COLS = [
    'trans_num',
    'DistanceFromHome', 
    'DistanceFromLastTxn', 
    'ImpossibleTravelFlag',
    'AmountDeviation_30d', 
    'SpendBurstScore', 
    'VelocityAcceleration',
    'TimeOfDayDeviation', 
    'CategoryDeviationScore',
    'InterArrivalEntropy', 
    'MerchantAmountZScore',
    'RiskSignalWeightedScore',
    'NullPwdChange', 
    'HasPasswordChange', 
    'DaysSincePasswordChangeBucket',
    'DeviceClusterSize', 
    'LinkedFraudAccounts', 
    'SharedMerchantFraudRate',
    'log_amt', 
    'log_DistanceFromHome', 
    'log_TimeSinceLastTxn', 
    'log_TxnAmount_7d',
    'IsNewDevice_x_IPRisk', 
    'IsVPN_x_AmtZScore', 
    'Bot_x_Velocity',
]

out = df[PYTHON_FEATURE_COLS + ['is_fraud']].copy()

print(f"{len(PYTHON_FEATURE_COLS)} features:")
out.describe().T[['mean','std','min','max']]

25 features:


,mean,std,min,max
trans_num,5.5,3.605551,0.0,11.0
DistanceFromHome,89.166667,151.15074,2.38,367.5
DistanceFromLastTxn,89.1075,152.508368,0.0,365.31
ImpossibleTravelFlag,0.0,0.0,0.0,0.0
AmountDeviation_30d,15.027883,52.332018,-1.0667,181.2013
SpendBurstScore,88.716958,24.231354,25.6271,100.0
VelocityAcceleration,20.5,5.664884,8.0,24.0
TimeOfDayDeviation,0.0,0.904537,-1.4415,1.5
CategoryDeviationScore,0.625,0.130558,0.5,0.75
InterArrivalEntropy,0.9183,0.0,0.9183,0.9183
